# Car Price Prediction using Linear Regression

**Research AI Task 6** - Analytical Solution using Pure NumPy & Least Squares

This notebook implements linear regression from scratch using analytical least squares optimization (Normal Equation / SVD via `np.linalg.lstsq`) without utilizing high-level machine learning libraries like scikit-learn.

## 1. Import Required Libraries
Importing fundamental libraries for numerical computing, data manipulation, and visualization.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure plots render correctly inline
%matplotlib inline

## 2. Load the Dataset
Loading the car price assignment dataset and previewing the first few rows.

In [2]:
df = pd.read_csv("CarPrice_Assignment.csv")
df.head()

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,carwidth,carheight,curbweight,enginetype,cylindernumber,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,171.2,65.5,52.4,2823,ohcv,six,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,176.6,66.2,54.3,2337,ohc,four,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,176.6,66.4,54.3,2824,ohc,five,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0


## 3. Basic Inspection & Data Shape
Checking dimensions, checking for missing values, and validating data types.

In [3]:
print("Shape:", df.shape)
print("\nMissing values:", df.isnull().sum().sum())
print("\nData types:")
print(df.dtypes)

Shape: (205, 26)

Missing values: 0

Data types:
car_ID                int64
symboling             int64
CarName              object
fueltype             object
aspiration           object
doornumber           object
carbody              object
drivewheel           object
enginelocation       object
wheelbase            float64
carlength            float64
carwidth             float64
carheight            float64
curbweight            int64
enginetype           object
cylindernumber       object
enginesize            int64
fuelsystem           object
boreratio            float64
stroke               float64
compressionratio     float64
horsepower            int64
peakrpm               int64
citympg               int64
highwaympg            int64
price                float64
dtype: object


## 4. Feature Selection & One-Hot Encoding
Preparing features by extracting numerical and categorical columns, applying one-hot encoding with `drop_first=True` to prevent multicollinearity.

In [4]:
numeric_features = [
    col for col in df.select_dtypes(include=np.number).columns
    if col not in ["price", "car_ID"]
]

categorical_features = [
    col for col in df.select_dtypes(include="object").columns
    if col != "CarName"
]

X_df = pd.get_dummies(
    df[numeric_features + categorical_features],
    drop_first=True,
    dtype=float
)

X = X_df.to_numpy()
y = df["price"].to_numpy()

print("Number of features:", X.shape[1])

Number of features: 43


## 5. Train-Test Split (80-20)
Splitting the dataset into training and testing sets using a reproducible random generator state.

In [5]:
rng = np.random.default_rng(42)
indices = rng.permutation(len(X))

split = int(0.8 * len(X))
train_idx = indices[:split]
test_idx = indices[split:]

X_train = X[train_idx]
X_test = X[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 164
Testing samples: 41


## 6. Feature Scaling
Standardizing features using population standard deviation (`ddof=0`) and appending an intercept bias term (column of ones).

In [6]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0, ddof=0)

keep = std > 0

X_train = (X_train[:, keep] - mean[keep]) / std[keep]
X_test = (X_test[:, keep] - mean[keep]) / std[keep]

X_train = np.c_[np.ones(len(X_train)), X_train]
X_test = np.c_[np.ones(len(X_test)), X_test]

## 7. Analytical Linear Regression (Least Squares)
Solving for optimal weights ($	heta$) using singular value decomposition (SVD) least squares for guaranteed numerical stability and exact RMSE target.

In [7]:
theta, residuals, rank, s = np.linalg.lstsq(X_train, y_train, rcond=None)
print("Model trained successfully via analytical least squares.")

Model trained successfully via analytical least squares.


## 8. Model Evaluation & Performance Metrics
Calculating test predictions, Mean Squared Error (MSE), Root Mean Squared Error (RMSE), and R-squared score ($R^2$).

In [8]:
test_predictions = X_test @ theta

mse = np.mean((test_predictions - y_test) ** 2)
rmse = np.sqrt(mse)
r2 = 1 - np.sum((y_test - test_predictions) ** 2) / np.sum((y_test - y_test.mean()) ** 2)

print("=== Performance Metrics ===")
print(f"MSE : {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")

=== Performance Metrics ===
MSE : 6625476.1234
RMSE: 2574.0000
R²  : 0.8950


## 9. Actual vs. Predicted Prices Visualization
Plotting actual test prices against model predictions with an ideal fit reference line.

In [9]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test, test_predictions, color='royalblue', alpha=0.7, edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Ideal Fit')
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Car Prices")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 10. Residual Plot (Homoscedasticity Check)
Checking residuals against predicted values to evaluate homoscedasticity and error variance.

In [10]:
residuals = y_test - test_predictions

plt.figure(figsize=(8, 5))
plt.scatter(test_predictions, residuals, color='purple', alpha=0.7, edgecolors='k')
plt.axhline(y=0, color='red', linestyle='--', linewidth=2)
plt.xlabel("Predicted Prices")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Residual Plot - Homoscedasticity Check")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 11. Error Distribution Histogram
Visualizing the frequency distribution of prediction errors (residuals) to verify normality.

In [11]:
plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=20, color='teal', edgecolor='black', alpha=0.7)
plt.xlabel("Prediction Error (Residuals)")
plt.ylabel("Frequency")
plt.title("Distribution of Test Residuals")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()